## BATCH NORMALIZATION

In [1]:
import numpy as np
import torch
import torch.nn as nn

## BatchNorm1d

In [2]:
# Configuración de dimensiones
m = 4             # Número de ejemplos en el minibatch 
hidden = 5        # Número de neuronas de la capa actual 
hidden_prev = 6   # Activaciones de la capa previa o número de features

# Creación de datos aleatorios
W = np.random.randn(hidden, hidden_prev) 
Z_prev = np.random.randint(0, 100, size=( m, hidden_prev)) 

# Inicialización de parámetros de Batch Normalization
gamma = np.ones((hidden, 1))  # Normalmente se inicia en 1 
beta = np.zeros((hidden, 1))   # Normalmente se inicia en 0 

In [3]:
# Cálculo de la salida lineal Z
Z = W @ Z_prev.T

In [4]:
Z

array([[  77.22572782,  -27.768714  ,   91.92816629,  -20.45856459],
       [ -12.6557553 ,  -79.68451648,  -39.62156518,   20.93570652],
       [  39.94192018,   52.41398172,   71.48094606,    4.43896866],
       [-160.02038838, -109.06718986, -179.4689984 ,   15.73893779],
       [ -10.22977838, -163.64750241,   43.30597633,  -49.82901967]])

In [5]:
def batch_norm(Z, gamma, beta):
    mean = np.mean(Z, axis=1, keepdims=True)
    std_dev = np.std(Z, axis=1, keepdims=True)
    Z_ = (Z - mean) / (std_dev + 1e-8)
    return gamma * Z_ + beta

In [6]:
Z_normalized = batch_norm(Z, gamma, beta)
print("Salida normalizada Z:\n", Z_normalized)

Salida normalizada Z:
 [[ 0.85983956 -1.06121914  1.12884663 -0.92746705]
 [ 0.40961807 -1.40857917 -0.32184646  1.32080756]
 [-0.08696953  0.42298438  1.2025887  -1.53860356]
 [-0.68146136 -0.01134691 -0.93724107  1.63004934]
 [ 0.45878579 -1.55971896  1.16315144 -0.06221827]]


In [7]:
print(Z.mean(axis=1, keepdims=True))  # Media a través de las columnas
print(Z.std(axis=1, keepdims=True))   # Desviación estándar a través

[[  30.23165388]
 [ -27.75653261]
 [  42.06895415]
 [-108.20440971]
 [ -45.10008103]]
[[54.65446814]
 [36.86550601]
 [24.4572328 ]
 [76.03656197]
 [76.00562966]]


In [8]:
def batch_norm_pytorch(Z, hidden):
    # PyTorch asume dimensiones (N, C), por eso transponemos Z
    Z_tensor = torch.tensor(Z.T, dtype=torch.float32)
    
    # Creamos el objeto de Batch Normalization de PyTorch
    bn = nn.BatchNorm1d(hidden)
    
    # Calculamos y regresamos a la forma original (transponiendo de nuevo)
    output = bn(Z_tensor)
    return output.T 

In [9]:
# Ejecución de la función manual
Z_normalized_pytorch = batch_norm_pytorch(Z, hidden).detach().numpy()
print("Salida normalizada con PyTorch:\n", Z_normalized_pytorch)


Salida normalizada con PyTorch:
 [[ 0.8598395  -1.0612192   1.1288465  -0.9274671 ]
 [ 0.40961805 -1.4085792  -0.3218465   1.3208075 ]
 [-0.08696949  0.4229844   1.2025888  -1.5386035 ]
 [-0.6814614  -0.01134698 -0.9372411   1.6300493 ]
 [ 0.45878583 -1.559719    1.1631515  -0.06221822]]


In [10]:
np.allclose(Z_normalized, Z_normalized_pytorch)

True

## BatchNorm2d

In [25]:
# Dimensiones de ejemplo: 5x5 (espacial), 4 canales, batch de 8
H, W, C, M = 5, 5, 4, 8

# Generación de activaciones aleatorias (Z)
Z = np.random.randint(0, 100, (H, W, C, M))

# Inicialización de parámetros Gamma y Beta con dimensiones para broadcasting
# Se mantienen las dimensiones 1, 1, C, 1 para operar directamente con Z 
gamma_2d = np.ones((1, 1, C, 1))
beta_2d = np.zeros((1, 1, C, 1))

In [26]:
def batch_norm_2d_manual(Z, gamma, beta):
    # Calculamos media y desviación estándar promediando H, W y el minibatch (ejes 0, 1 y 3)
    mu = np.mean(Z, axis=(0, 1, 3), keepdims=True)
    std = np.std(Z, axis=(0, 1, 3), keepdims=True)
    
    # Normalización con un epsilon para estabilidad numérica
    epsilon = 1e-8
    Z_norm = (Z - mu) / (std + epsilon)
    
    # Aplicar escala (gamma) y desplazamiento (beta)
    return gamma * Z_norm + beta

In [27]:
Z_normalized_2d = batch_norm_2d_manual(Z, gamma_2d, beta_2d)
Z_normalized_2d.shape, Z.shape 

((5, 5, 4, 8), (5, 5, 4, 8))

In [28]:
def batch_norm_2d_pytorch(Z, channels):
    # 1. Transponer de (H, W, C, M) a (M, C, H, W) que es lo que espera PyTorch
    # Posiciones originales: 0:H, 1:W, 2:C, 3:M -> Nueva orden: 3, 2, 0, 1
    Z_tensor = torch.tensor(Z, dtype=torch.float32).permute(3, 2, 0, 1)
    
    # 2. Crear y aplicar la capa de Batch Normalization 2D
    bn = nn.BatchNorm2d(channels)
    output = bn(Z_tensor)
    
    # 3. Regresar a las dimensiones originales (H, W, C, M) para comparar
    # De (M, C, H, W) -> Posiciones: 0:M, 1:C, 2:H, 3:W -> Queremos 2, 3, 1, 0
    output_original_shape = output.permute(2, 3, 1, 0)
    
    return output_original_shape.detach().numpy()


In [29]:
Z_normalized_pytorch = batch_norm_2d_pytorch(Z, C)
Z_normalized_pytorch.shape

(5, 5, 4, 8)

In [30]:
np.allclose(Z_normalized_2d, Z_normalized_pytorch)

True